<a href="https://colab.research.google.com/github/kavyaayyappan1998/DATA266---2738/blob/main/cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Personal parameters
SID4 = 2738
SEED = 2738
SLICE = 738
HP_ID = 2
CLS_A = 8
CLS_B = 2

In [2]:
!nvidia-smi
!nvcc --version

Tue Sep  1 06:48:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Implement matrix multiplication using CUDA C programming


#### Use blocks and threads for parallel computation, clearly explain the blocks and threads in the code.
#### Profile the CUDA program using Nsight Systems, Nsight Compute, or nvprof if it is available in your environment. State which profiler you used.


In [7]:
%%writefile matrix_mul.cu
#include <cuda_runtime.h>
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <random>
#include <string>
#include <vector>

#define TILE 16
#define CPU_TILE 32

const unsigned int SEED = 2738;
const int CPU_REPEATS = 3;
const int GPU_REPEATS = 5;

// Simple CUDA error check
#define CUDA_CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        std::cerr << "CUDA error: " << cudaGetErrorString(err) << std::endl; \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)

// CUDA matrix multiplication kernel
// Each thread calculates one value in output matrix C
__global__ void matrixMultiplyGPU(
    const float* A,
    const float* B,
    float* C,
    int N
) {
    __shared__ float tileA[TILE][TILE];
    __shared__ float tileB[TILE][TILE];

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    float sum = 0.0f;

    int numberOfTiles = (N + TILE - 1) / TILE;

    for (int t = 0; t < numberOfTiles; t++) {
        int aCol = t * TILE + threadIdx.x;
        int bRow = t * TILE + threadIdx.y;

        if (row < N && aCol < N)
            tileA[threadIdx.y][threadIdx.x] =
                A[(size_t)row * N + aCol];
        else
            tileA[threadIdx.y][threadIdx.x] = 0.0f;

        if (bRow < N && col < N)
            tileB[threadIdx.y][threadIdx.x] =
                B[(size_t)bRow * N + col];
        else
            tileB[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        for (int k = 0; k < TILE; k++) {
            sum += tileA[threadIdx.y][k] *
                   tileB[k][threadIdx.x];
        }

        __syncthreads();
    }

    if (row < N && col < N) {
        C[(size_t)row * N + col] = sum;
    }
}

// CPU matrix multiplication used as baseline
void matrixMultiplyCPU(
    const std::vector<float>& A,
    const std::vector<float>& B,
    std::vector<float>& C,
    int N
) {
    std::fill(C.begin(), C.end(), 0.0f);

    #pragma omp parallel for collapse(2) schedule(static)
    for (int ii = 0; ii < N; ii += CPU_TILE) {
        for (int jj = 0; jj < N; jj += CPU_TILE) {

            int iEnd = std::min(ii + CPU_TILE, N);
            int jEnd = std::min(jj + CPU_TILE, N);

            for (int kk = 0; kk < N; kk += CPU_TILE) {

                int kEnd = std::min(kk + CPU_TILE, N);

                for (int i = ii; i < iEnd; i++) {
                    for (int k = kk; k < kEnd; k++) {

                        float a = A[(size_t)i * N + k];

                        for (int j = jj; j < jEnd; j++) {
                            C[(size_t)i * N + j] +=
                                a * B[(size_t)k * N + j];
                        }
                    }
                }
            }
        }
    }
}

struct GPUTiming {
    double kernel_ms;
    double transfer_ms;
};

// Fill matrix with random values
void fillRandom(
    std::vector<float>& values,
    std::mt19937& rng
) {
    std::uniform_real_distribution<float> dist(-1.0f, 1.0f);

    for (float& value : values) {
        value = dist(rng);
    }
}

// Check CPU and GPU results
double maxAbsError(
    const std::vector<float>& A,
    const std::vector<float>& B
) {
    double maxError = 0.0;

    for (size_t i = 0; i < A.size(); i++) {
        maxError = std::max(
            maxError,
            (double)std::fabs(A[i] - B[i])
        );
    }

    return maxError;
}

// Time CPU version
double timeCPU(
    const std::vector<float>& A,
    const std::vector<float>& B,
    std::vector<float>& C,
    int N
) {
    // Small warm-up
    int W = 128;

    std::vector<float> warmA((size_t)W * W, 1.0f);
    std::vector<float> warmB((size_t)W * W, 1.0f);
    std::vector<float> warmC((size_t)W * W, 0.0f);

    matrixMultiplyCPU(warmA, warmB, warmC, W);

    double total = 0.0;

    for (int r = 0; r < CPU_REPEATS; r++) {

        auto start =
            std::chrono::high_resolution_clock::now();

        matrixMultiplyCPU(A, B, C, N);

        auto end =
            std::chrono::high_resolution_clock::now();

        total +=
            std::chrono::duration<double, std::milli>(
                end - start
            ).count();
    }

    return total / CPU_REPEATS;
}

// Time GPU kernel and transfer separately
GPUTiming timeGPU(
    const std::vector<float>& A,
    const std::vector<float>& B,
    std::vector<float>& C,
    int N,
    int repeats
) {
    size_t elements = (size_t)N * N;
    size_t bytes = elements * sizeof(float);

    float* dA = nullptr;
    float* dB = nullptr;
    float* dC = nullptr;

    CUDA_CHECK(cudaMalloc(&dA, bytes));
    CUDA_CHECK(cudaMalloc(&dB, bytes));
    CUDA_CHECK(cudaMalloc(&dC, bytes));

    dim3 threads(TILE, TILE);

    dim3 blocks(
        (N + TILE - 1) / TILE,
        (N + TILE - 1) / TILE
    );

    // GPU warm-up
    CUDA_CHECK(
        cudaMemcpy(
            dA,
            A.data(),
            bytes,
            cudaMemcpyHostToDevice
        )
    );

    CUDA_CHECK(
        cudaMemcpy(
            dB,
            B.data(),
            bytes,
            cudaMemcpyHostToDevice
        )
    );

    matrixMultiplyGPU<<<blocks, threads>>>(
        dA,
        dB,
        dC,
        N
    );

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    cudaEvent_t startEvent;
    cudaEvent_t stopEvent;

    CUDA_CHECK(cudaEventCreate(&startEvent));
    CUDA_CHECK(cudaEventCreate(&stopEvent));

    double kernelTotal = 0.0;
    double transferTotal = 0.0;

    for (int r = 0; r < repeats; r++) {

        float h2d = 0.0f;
        float kernel = 0.0f;
        float d2h = 0.0f;

        // Host to device transfer
        CUDA_CHECK(cudaEventRecord(startEvent));

        CUDA_CHECK(
            cudaMemcpy(
                dA,
                A.data(),
                bytes,
                cudaMemcpyHostToDevice
            )
        );

        CUDA_CHECK(
            cudaMemcpy(
                dB,
                B.data(),
                bytes,
                cudaMemcpyHostToDevice
            )
        );

        CUDA_CHECK(cudaEventRecord(stopEvent));
        CUDA_CHECK(cudaEventSynchronize(stopEvent));

        CUDA_CHECK(
            cudaEventElapsedTime(
                &h2d,
                startEvent,
                stopEvent
            )
        );

        // GPU kernel time
        CUDA_CHECK(cudaEventRecord(startEvent));

        matrixMultiplyGPU<<<blocks, threads>>>(
            dA,
            dB,
            dC,
            N
        );

        CUDA_CHECK(cudaGetLastError());

        CUDA_CHECK(cudaEventRecord(stopEvent));
        CUDA_CHECK(cudaEventSynchronize(stopEvent));

        CUDA_CHECK(
            cudaEventElapsedTime(
                &kernel,
                startEvent,
                stopEvent
            )
        );

        // Device to host transfer
        CUDA_CHECK(cudaEventRecord(startEvent));

        CUDA_CHECK(
            cudaMemcpy(
                C.data(),
                dC,
                bytes,
                cudaMemcpyDeviceToHost
            )
        );

        CUDA_CHECK(cudaEventRecord(stopEvent));
        CUDA_CHECK(cudaEventSynchronize(stopEvent));

        CUDA_CHECK(
            cudaEventElapsedTime(
                &d2h,
                startEvent,
                stopEvent
            )
        );

        kernelTotal += kernel;
        transferTotal += h2d + d2h;
    }

    CUDA_CHECK(cudaEventDestroy(startEvent));
    CUDA_CHECK(cudaEventDestroy(stopEvent));

    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));

    GPUTiming result;

    result.kernel_ms =
        kernelTotal / repeats;

    result.transfer_ms =
        transferTotal / repeats;

    return result;
}

// Run one matrix size
void runTest(int N) {

    size_t elements = (size_t)N * N;

    std::vector<float> A(elements);
    std::vector<float> B(elements);
    std::vector<float> Ccpu(elements, 0.0f);
    std::vector<float> Cgpu(elements, 0.0f);

    std::mt19937 rng(SEED + N);

    fillRandom(A, rng);
    fillRandom(B, rng);

    double cpuTime =
        timeCPU(A, B, Ccpu, N);

    GPUTiming gpu =
        timeGPU(
            A,
            B,
            Cgpu,
            N,
            GPU_REPEATS
        );

    double gpuEndToEnd =
        gpu.kernel_ms + gpu.transfer_ms;

    double speedup =
        cpuTime / gpuEndToEnd;

    double error =
        maxAbsError(Ccpu, Cgpu);

    std::cout
        << std::fixed
        << std::setprecision(4)
        << N << ","
        << cpuTime << ","
        << gpu.kernel_ms << ","
        << gpu.transfer_ms << ","
        << gpuEndToEnd << ","
        << speedup << ","
        << error
        << std::endl;
}

// Used only for profiler
void runGPUOnly(int N) {

    size_t elements = (size_t)N * N;

    std::vector<float> A(elements);
    std::vector<float> B(elements);
    std::vector<float> C(elements, 0.0f);

    std::mt19937 rng(SEED + N);

    fillRandom(A, rng);
    fillRandom(B, rng);

    GPUTiming gpu =
        timeGPU(A, B, C, N, 1);

    std::cout
        << "GPU_ONLY,"
        << "N=" << N << ","
        << "kernel_ms=" << gpu.kernel_ms << ","
        << "H2D_plus_D2H_ms=" << gpu.transfer_ms
        << std::endl;
}

int main(int argc, char** argv) {

    CUDA_CHECK(cudaSetDevice(0));

    cudaDeviceProp prop;

    CUDA_CHECK(
        cudaGetDeviceProperties(
            &prop,
            0
        )
    );

    std::cout
        << "GPU="
        << prop.name
        << std::endl;

    std::cout
        << "SEED="
        << SEED
        << std::endl;

    std::cout
        << "BLOCK=16x16=256 threads"
        << std::endl;

    // Run only one size if requested
    if (
        argc == 3 &&
        std::string(argv[1]) == "--size"
    ) {

        std::cout
            << "N,CPU_ms,GPU_kernel_ms,"
               "H2D_plus_D2H_ms,"
               "GPU_end_to_end_ms,"
               "Speedup,MaxAbsError"
            << std::endl;

        runTest(
            std::stoi(argv[2])
        );

        return 0;
    }

    // Used for profiler
    if (
        argc == 3 &&
        std::string(argv[1]) == "--gpu-only"
    ) {

        runGPUOnly(
            std::stoi(argv[2])
        );

        return 0;
    }

    std::cout
        << "N,CPU_ms,GPU_kernel_ms,"
           "H2D_plus_D2H_ms,"
           "GPU_end_to_end_ms,"
           "Speedup,MaxAbsError"
        << std::endl;

    for (int N : {256, 1024, 4096}) {
        runTest(N);
    }

    return 0;
}


Overwriting matrix_mul.cu


In [8]:
# Check that the .cu file was created
!ls -lh matrix_mul.cu

-rw-r--r-- 1 root root 11K Sep  1 06:51 matrix_mul.cu


### Compile the CUDA code


In [9]:
!nvcc -O3 -Xcompiler -fopenmp -Wno-deprecated-gpu-targets matrix_mul.cu -o matrix_mul

# Check that compilation worked
!ls -lh matrix_mul


-rwxr-xr-x 1 root root 999K Sep  1 06:51 matrix_mul


### Test the program with matrix size 256


In [10]:
!./matrix_mul --size 256


GPU=Tesla T4
SEED=2738
BLOCK=16x16=256 threads
N,CPU_ms,GPU_kernel_ms,H2D_plus_D2H_ms,GPU_end_to_end_ms,Speedup,MaxAbsError
256,12.5235,0.0808,0.2902,0.3710,33.7554,0.0000


# Measurement and analysis


## 4. CUDA

CUDA: time your kernel against a CPU baseline for matrix sizes 256, 1024, and 4096. Report CPU time, GPU kernel time, host-to-device plus device-to-host transfer time, and end-to-end speedup. Include profiler output that separates kernel time from transfer time.


In [11]:
import subprocess
import pandas as pd
from datetime import datetime

sizes = [256, 1024, 4096]

results = []
log_text = []

start_time = datetime.now()

print("Benchmark started:", start_time)

for N in sizes:

    print(f"\nRunning matrix size {N}")

    run = subprocess.run(
        ["./matrix_mul", "--size", str(N)],
        capture_output=True,
        text=True,
        check=True
    )

    print(run.stdout)

    log_text.append(
        f"===== N={N} =====\n"
        + run.stdout
        + run.stderr
    )

    result_line = None

    for line in run.stdout.splitlines():
        if line.startswith(f"{N},"):
            result_line = line
            break

    if result_line is None:
        raise RuntimeError(
            f"No result found for N={N}"
        )

    values = result_line.split(",")

    results.append({
        "Matrix size": int(values[0]),
        "CPU (ms)": float(values[1]),
        "GPU kernel (ms)": float(values[2]),
        "H2D+D2H (ms)": float(values[3]),
        "GPU end-to-end (ms)": float(values[4]),
        "Speedup": float(values[5]),
        "MaxAbsError": float(values[6])
    })

benchmark_df = pd.DataFrame(results)

print("\nFinal results")
display(benchmark_df)

benchmark_df.to_csv(
    "cuda_metrics.csv",
    index=False
)

with open(
    "RUN_LOG_cuda.txt",
    "w"
) as f:

    f.write(
        "DATA 266 HW1 CUDA run\n"
    )

    f.write(
        f"SID4 = {SID4}\n"
    )

    f.write(
        f"SEED = {SEED}\n"
    )

    f.write(
        f"Start time = {start_time}\n\n"
    )

    f.write(
        "\n".join(log_text)
    )

print("Saved cuda_metrics.csv")
print("Saved RUN_LOG_cuda.txt")


Benchmark started: 2026-09-01 06:51:37.289922

Running matrix size 256
GPU=Tesla T4
SEED=2738
BLOCK=16x16=256 threads
N,CPU_ms,GPU_kernel_ms,H2D_plus_D2H_ms,GPU_end_to_end_ms,Speedup,MaxAbsError
256,3.9078,0.0673,0.2504,0.3177,12.2996,0.0000


Running matrix size 1024
GPU=Tesla T4
SEED=2738
BLOCK=16x16=256 threads
N,CPU_ms,GPU_kernel_ms,H2D_plus_D2H_ms,GPU_end_to_end_ms,Speedup,MaxAbsError
1024,294.6008,2.8759,2.8233,5.6991,51.6924,0.0000


Running matrix size 4096
GPU=Tesla T4
SEED=2738
BLOCK=16x16=256 threads
N,CPU_ms,GPU_kernel_ms,H2D_plus_D2H_ms,GPU_end_to_end_ms,Speedup,MaxAbsError
4096,24184.6447,197.3910,44.4779,241.8689,99.9907,0.0001


Final results


,Matrix size,CPU (ms),GPU kernel (ms),H2D+D2H (ms),GPU end-to-end (ms),Speedup,MaxAbsError
0,256,3.9078,0.0673,0.2504,0.3177,12.2996,0.0000
1,1024,294.6008,2.8759,2.8233,5.6991,51.6924,0.0000
2,4096,24184.6447,197.3910,44.4779,241.8689,99.9907,0.0001


Saved cuda_metrics.csv
Saved RUN_LOG_cuda.txt


### Required CUDA table


In [12]:
table = benchmark_df[
    [
        "Matrix size",
        "CPU (ms)",
        "GPU kernel (ms)",
        "H2D+D2H (ms)",
        "Speedup"
    ]
]

display(table)

print("\nMarkdown table:")
print(table.to_markdown(index=False))


,Matrix size,CPU (ms),GPU kernel (ms),H2D+D2H (ms),Speedup
0,256,3.9078,0.0673,0.2504,12.2996
1,1024,294.6008,2.8759,2.8233,51.6924
2,4096,24184.6447,197.3910,44.4779,99.9907



Markdown table:
|   Matrix size |   CPU (ms) |   GPU kernel (ms) |   H2D+D2H (ms) |   Speedup |
|--------------:|-----------:|------------------:|---------------:|----------:|
|           256 |     3.9078 |            0.0673 |         0.2504 |   12.2996 |
|          1024 |   294.601  |            2.8759 |         2.8233 |   51.6924 |
|          4096 | 24184.6    |          197.391  |        44.4779 |   99.9907 |


## CUDA profiler

The question says:

**Include profiler output that separates kernel time from transfer time.**

I first check if Nsight Systems, Nsight Compute, or nvprof is available in Colab.


In [15]:
import shutil
from pathlib import Path

def find_tool(name):

    path = shutil.which(name)

    if path:
        return path

    possible_paths = [
        f"/usr/local/cuda/bin/{name}",
        f"/usr/local/bin/{name}",
        f"/opt/nvidia/nsight-systems/bin/{name}"
    ]

    for path in possible_paths:

        if Path(path).exists():
            return path

    return None


profiler_paths = {
    "nsys": find_tool("nsys"),
    "ncu": find_tool("ncu"),
    "nvprof": find_tool("nvprof")
}

print(profiler_paths)


{'nsys': None, 'ncu': '/usr/local/cuda/bin/ncu', 'nvprof': '/usr/local/cuda/bin/nvprof'}


In [17]:
# Use the first profiler that is available

import subprocess

profiler_used = None
profiler_output = ""

if profiler_paths["nsys"]:

    profiler_used = "Nsight Systems"

    command = [
        profiler_paths["nsys"],
        "profile",
        "--trace=cuda",
        "--stats=true",
        "--force-overwrite=true",
        "-o",
        "hw1_profile",
        "./matrix_mul",
        "--gpu-only",
        "1024"
    ]

elif profiler_paths["ncu"]:

    profiler_used = "Nsight Compute"

    command = [
        profiler_paths["ncu"],
        "--set",
        "basic",
        "./matrix_mul",
        "--gpu-only",
        "1024"
    ]

elif profiler_paths["nvprof"]:

    profiler_used = "nvprof"

    command = [
        profiler_paths["nvprof"],
        "./matrix_mul",
        "--gpu-only",
        "1024"
    ]

else:

    command = None


if command is None:

    print(
        "No Nsight Systems, Nsight Compute, "
        "or nvprof was available in this Colab runtime."
    )

    profiler_output = (
        "No approved profiler was available."
    )

else:

    print("Profiler used:", profiler_used)

    run = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    profiler_output = (
        run.stdout
        + "\n"
        + run.stderr
    )

    print(profiler_output)


with open(
    "profiler_output.txt",
    "w"
) as f:

    f.write(
        f"Profiler used: {profiler_used}\n\n"
    )

    f.write(
        profiler_output
    )


Profiler used: Nsight Compute
==PROF== Connected to process 2533 (/content/matrix_mul)
GPU=Tesla T4
SEED=2738
BLOCK=16x16=256 threads
==PROF== Profiling "matrixMultiplyGPU" - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "matrixMultiplyGPU" - 1: 0%....50%....100% - 9 passes
GPU_ONLY,N=1024,kernel_ms=1669.46,H2D_plus_D2H_ms=3.28291
==PROF== Disconnected from process 2533
[2533] matrix_mul@127.0.0.1
  matrixMultiplyGPU(const float *, const float *, float *, int) (64, 64, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycle    3,380,466
    Memory Throughput                 %        74.59
    DRAM Throughput                   %        10.77


## 5. GPU end-to-end crossover

#### In three or four sentences, identify the smallest matrix size at which GPU end-to-end time becomes beneficial in your measurements and explain why the crossover is not at size zero.**

The next cell uses my measured results and creates the answer.


In [19]:
faster_gpu = benchmark_df[
    benchmark_df["Speedup"] > 1.0
].sort_values("Matrix size")

if len(faster_gpu) > 0:

    row = faster_gpu.iloc[0]

    N = int(
        row["Matrix size"]
    )

    crossover_text = (
        f"In my measurements, the GPU first became faster "
        f"at matrix size {N} x {N}. "
        f"The CPU time was {row['CPU (ms)']:.2f} ms and the "
        f"GPU end-to-end time was "
        f"{row['GPU end-to-end (ms)']:.2f} ms, giving a speedup "
        f"of {row['Speedup']:.2f}x. "
        f"The crossover is not at size zero because the GPU has "
        f"extra overhead from kernel launch and data transfer. "
        f"For larger matrices, the GPU has enough parallel work "
        f"to make up for this overhead."
    )

else:

    crossover_text = (
        "In my measurements, the GPU was not faster end-to-end "
        "for any of the tested sizes 256, 1024, and 4096. "
        "The GPU has extra overhead from kernel launch and "
        "host-to-device and device-to-host data transfer. "
        "For small workloads, this overhead can be larger than "
        "the benefit from parallel processing."
    )
print(crossover_text)

In my measurements, the GPU first became faster at matrix size 256 x 256. The CPU time was 3.91 ms and the GPU end-to-end time was 0.32 ms, giving a speedup of 12.30x. The crossover is not at size zero because the GPU has extra overhead from kernel launch and data transfer. For larger matrices, the GPU has enough parallel work to make up for this overhead.


## Save CUDA measurements


In [20]:
metrics_text = (
    "# HW1 CUDA Metrics\n\n"
    + table.to_markdown(index=False)
    + "\n\n"
    + "## GPU crossover\n\n"
    + crossover_text
    + "\n"
)

with open(
    "METRICS_cuda.md",
    "w"
) as f:

    f.write(
        metrics_text
    )

print(metrics_text)


# HW1 CUDA Metrics

|   Matrix size |   CPU (ms) |   GPU kernel (ms) |   H2D+D2H (ms) |   Speedup |
|--------------:|-----------:|------------------:|---------------:|----------:|
|           256 |     3.9078 |            0.0673 |         0.2504 |   12.2996 |
|          1024 |   294.601  |            2.8759 |         2.8233 |   51.6924 |
|          4096 | 24184.6    |          197.391  |        44.4779 |   99.9907 |

## GPU crossover

In my measurements, the GPU first became faster at matrix size 256 x 256. The CPU time was 3.91 ms and the GPU end-to-end time was 0.32 ms, giving a speedup of 12.30x. The crossover is not at size zero because the GPU has extra overhead from kernel launch and data transfer. For larger matrices, the GPU has enough parallel work to make up for this overhead.

